# Assgn_014 — Token ladder (smoke → 2M → 5M → 10M → optional 50M)

Dense train → MoE upcycle → MoE continue. Each ladder step splits total budget **50/50** (`tokens_dense = T//2`, `tokens_moe = T//2`).

**Routing:** three-tier loss-free bias + probabilistic warm-up that **auto-extends to cover the MoE phase** (floor 2000; tag `wu_phase`). Aim for `dead_experts→0` before full. Smoke `all_ok` requires dead==0. New folders look like `vNNN_colab_t4_50m_full_wu_phase`. Do not RESUME failed `v015_…_wu2k`.

**CE floor:** vocab=8192 → random-token floor ≈ `ln(8192)≈9.011`. Tiny late loss drops are expected on synthetic data.

## Steps

1. Upload `Assign_0014_colab.zip` to **`/content/`** (not `/content/content/`).
2. **Runtime → Change runtime type → Hardware accelerator → GPU (T4)** — then Connect. Confirm with `!nvidia-smi` (should show Tesla T4). Re-run later cells only after GPU is attached.
3. Run **Cell 0** (unzip + pip).
4. Run **Cell 1** (setup, **batch=8**). Drive mount is attempted; if it fails, reports stay under `/content/Assign_0014/reports`.
5. Run **Smoke** (4k+4k) → confirm gate + PNGs + dead_experts trending → zip/download.
6. Run **2M** (1M+1M) → zip/download.
7. Run **5M** (2.5M+2.5M) → zip/download.
8. Run **10M** (5M+5M) → zip/download.
9. (Optional / graded) Run **50M** with `smoke=False` (25M+25M) → zip.
10. Run **Diagnostics** (`self_analyze` / `self_evaluate` / `self_correct`).

Keep `smoke=True` for 2/5/10M so the 25M+25M full gate does not false-FAIL. Do **not** set `RESUME` between different budgets (each gets a new `vNNN_...` folder).

**Zip hygiene:** download `/content/<version>.zip`, then extract so files land in `reports/<version>/` (manifest next to PNGs). Do **not** extract into a pre-created folder with the same name (that nests `v00N/.../v00N/...`).

**USD:** Colab T4 ≈ 0.12 USD/h. Free tier: set `ASSGN014_FREE_COLAB=1` in Cell 1.


## Cell 0 — unzip once

**Before this cell:** Runtime → Change runtime type → GPU (T4), then Connect.

Edit the zip path if your file landed elsewhere (e.g. `/content/content/Assign_0014_colab.zip`).


In [ ]:
# BEFORE this cell: Runtime → Change runtime type → Hardware accelerator → GPU (T4), then Connect.
# Optional GPU check (uncomment):
# !nvidia-smi

ZIP = '/content/Assign_0014_colab.zip'
# If upload went into a nested folder:
# ZIP = '/content/content/Assign_0014_colab.zip'

!unzip -q -o {ZIP} -d /content/Assign_0014
%cd /content/Assign_0014
!test -f .env || cp .env.example .env
!pip -q install -r requirements.txt
!pwd
!ls -la


## Cell 1 — setup + optional Drive

Drive mount often fails (popup blocked / timeout). This cell **catches** that and falls back to local `reports/`. Always zip after each budget if Drive is not mounted.

Sets **`ASSGN014_BATCH=8`** (T4 peak MoE was ~2 GB at B=2; headroom is large).


In [ ]:
from pathlib import Path
import os
import shutil
import sys

REPORTS_ROOT = None  # set below if Drive mounts
BATCH = 8  # T4 default; lower only if OOM

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    REPORTS_ROOT = '/content/drive/MyDrive/Assign_0014/reports'
    Path(REPORTS_ROOT).mkdir(parents=True, exist_ok=True)
    print('Drive OK → reports at', REPORTS_ROOT)
except Exception as exc:
    REPORTS_ROOT = None
    print('Drive mount failed — using local reports/ and zip-after-each.')
    print('Reason:', type(exc).__name__, str(exc)[:200])

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

env_path = ROOT / '.env'
example = ROOT / '.env.example'
if not env_path.is_file() and example.is_file():
    shutil.copy2(example, env_path)
    print('created .env from .env.example')

os.environ.setdefault('ASSGN014_HOST', 'colab')
os.environ.setdefault('ASSGN014_USD_PER_HOUR', '0.12')
os.environ['ASSGN014_BATCH'] = str(BATCH)
# Free Colab (no units): uncomment next two lines
# os.environ['ASSGN014_FREE_COLAB'] = '1'
# os.environ['ASSGN014_USD_PER_HOUR'] = '0.00'

if env_path.is_file():
    lines = []
    for line in env_path.read_text(encoding='utf-8').splitlines():
        if line.startswith('ASSGN014_HOST='):
            lines.append('ASSGN014_HOST=colab')
        elif line.startswith('ASSGN014_BATCH='):
            lines.append(f'ASSGN014_BATCH={BATCH}')
        elif line.startswith('ASSGN014_USD_PER_HOUR=') and os.environ.get('ASSGN014_USD_PER_HOUR'):
            lines.append(f"ASSGN014_USD_PER_HOUR={os.environ['ASSGN014_USD_PER_HOUR']}")
        else:
            lines.append(line)
    env_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

if REPORTS_ROOT:
    os.environ['ASSGN014_REPORTS_ROOT'] = REPORTS_ROOT
else:
    os.environ.pop('ASSGN014_REPORTS_ROOT', None)

from IPython.display import Image, display
from src.pipeline.moe_lab import run_all, format_gate_print
from src.utils.versioning import read_latest

reports_root = Path(os.environ['ASSGN014_REPORTS_ROOT']) if os.environ.get('ASSGN014_REPORTS_ROOT') else (ROOT / 'reports')
reports_root.mkdir(parents=True, exist_ok=True)


def zip_latest_results(results: dict) -> Path:
    """Zip version folder to /content/<version>.zip for download.

    Extract so files land in reports/<version>/ (not nested version/version/).
    """
    vid = results['version']
    vdir = Path(results['reports_dir'])
    out = Path('/content') / f'{vid}.zip'
    if out.exists():
        out.unlink()
    shutil.make_archive(str(out.with_suffix('')), 'zip', vdir)
    mb = round(out.stat().st_size / 1e6, 2)
    print(f'zipped {out} ({mb} MB) — download from Files pane before disconnect')
    print(f'Extract tip: unzip into reports/ so you get reports/{vid}/assgn014_manifest.json')
    return out


def show_primary(results: dict) -> None:
    vdir = Path(results['reports_dir'])
    for name in (
        'assgn014_loss_trajectories.png',
        'assgn014_expert_load_health.png',
        'assgn014_memory_speed_dashboard.png',
    ):
        p = vdir / name
        if p.is_file():
            display(Image(filename=str(p)))
        else:
            print('missing', p)


def run_ladder(total_tokens: int, *, smoke: bool = True, version=None):
    """50/50 dense+MoE split of total_tokens; batch from Cell 1."""
    td = int(total_tokens) // 2
    tm = int(total_tokens) - td
    return run_all(
        reports_root,
        root=ROOT,
        tokens_dense=td,
        tokens_moe=tm,
        batch=BATCH,
        smoke=smoke,
        version=version,
    )


print('ROOT', ROOT)
print('reports_root', reports_root)
print('BATCH', BATCH)
print('env_exists', env_path.is_file())


## Smoke (wiring check, 4k+4k)

Run this **before** 2M (and again after pulling biased/warm-up code). Confirm PNGs, zip path, and that `dead_experts` is trending down vs older flat-bias runs.


In [ ]:
results = run_all(
    reports_root,
    root=ROOT,
    tokens_dense=4096,
    tokens_moe=4096,
    batch=BATCH,
    smoke=True,
)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))
show_primary(results)
zip_latest_results(results)


## 2M ladder step

1M dense + 1M MoE. Then zip/download.


In [ ]:
results = run_ladder(2_000_000, smoke=True)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))
show_primary(results)
zip_latest_results(results)


## 5M ladder step

New version folder (do not RESUME the 2M id). 2.5M dense + 2.5M MoE.


In [ ]:
results = run_ladder(5_000_000, smoke=True)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))
show_primary(results)
zip_latest_results(results)


## 10M ladder step

5M dense + 5M MoE. Zip/download before starting the graded full run. Prefer `dead_experts` near 0 and MaxVio trending down before full.


In [ ]:
results = run_ladder(10_000_000, smoke=True)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))
show_primary(results)
zip_latest_results(results)


## Optional: graded 50M (25M dense + 25M MoE, hours on T4)

`smoke=False` enforces tokens ≥ 25M per phase **and** `dead_experts==0` with MaxVio≤2. Leave `RESUME = None` on first start. After a disconnect, set `RESUME` to the same `vNNN_...` id **only if** that folder still exists (Drive or re-uploaded zip).


In [ ]:
RESUME = None  # e.g. 'v008_colab_t4_full' after disconnect
if RESUME:
    os.environ['ASSGN014_RUN_VERSION'] = RESUME

results = run_all(
    reports_root,
    root=ROOT,
    tokens_dense=25_000_000,
    tokens_moe=25_000_000,
    batch=BATCH,
    smoke=False,
    version=RESUME,
)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))
show_primary(results)
zip_latest_results(results)


## Re-zip last results (if you skipped auto-zip)


In [ ]:
zip_latest_results(results)


## Diagnostics (self_analyze / evaluate / correct)

Run after smoke (balance check) and again after the graded full run.


In [ ]:
import subprocess

latest = read_latest(reports_root)
reports_arg = str(reports_root / latest) if latest else str(reports_root)
print('diagnostics on', reports_arg)
for script in ('self_analyze.py', 'self_evaluate.py', 'self_correct.py'):
    print('\n===', script, '===')
    rc = subprocess.call(
        [sys.executable, str(ROOT / 'tests' / script), '--reports', reports_arg],
        cwd=str(ROOT),
    )
    print('exit', rc)
